# Course 2 · Week 4 — Hands-on: Decision Trees from Scratch

By the end you'll have:

1. Implemented Shannon entropy
2. Implemented information gain — the rule for picking which feature to split on
3. Implemented a recursive tree builder
4. Trained a tree to perfectly classify a 10-row "cat or not cat" dataset
5. Visualized the tree structure
6. (Stretch) Built a 5-tree bagged forest and compared

Estimated time: **40–55 minutes.** Conceptually clean — recursion plus one log formula.


## Setup — the cat dataset

Ten animals, three yes/no features each. The labels in `y` say which were cats. Your tree's job: figure out the right *sequence of questions* that lets it correctly classify each animal.

This is small enough to fit in your head and verify the tree by hand if you want.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cat vs not-cat classification using three categorical features.
# 0/1 encoding so we can split with `==`.
# Feature 0: ear shape  (0 = floppy, 1 = pointy)
# Feature 1: face round (0 = no,     1 = yes)
# Feature 2: whiskers   (0 = no,     1 = yes)
X = np.array([
    [1, 1, 1],   # cat
    [0, 1, 1],   # cat
    [0, 0, 0],   # not cat
    [1, 0, 1],   # cat
    [1, 1, 1],   # cat
    [1, 1, 0],   # cat
    [0, 0, 0],   # not cat
    [1, 1, 0],   # cat
    [0, 1, 0],   # not cat
    [0, 1, 0],   # not cat
])
y = np.array([1, 1, 0, 1, 1, 1, 0, 1, 0, 0])

print(f"X shape {X.shape}")
print(f"Distribution: {(y==1).sum()} cats / {(y==0).sum()} not cats")


## Quick recap

A decision tree is a flowchart of yes/no questions. At each internal node, you ask one question; depending on the answer you go left or right. Leaves are the predictions.

The whole game is: **which question to ask first, and which next**. Algorithm: at every node, pick the feature whose split most reduces the *uncertainty* of the labels. We measure uncertainty with **entropy**, and reduction with **information gain**.

- Entropy of pure labels (all same class) = 0 (no uncertainty)
- Entropy of 50/50 labels = 1 (max uncertainty for binary)
- Info gain of a split = how much we reduced entropy by asking that question


## Exercise 1 — entropy

For binary labels with `p` = fraction of class 1:

$$H(p) = -p \log_2(p) - (1-p) \log_2(1-p)$$

Edge cases: pure (p = 0 or p = 1) → entropy 0 (`log(0)` would blow up; we handle by short-circuit).


In [ ]:
def entropy(y):
    """Shannon entropy of binary labels.

    For p = fraction of class 1:
      entropy = -p log2(p) - (1-p) log2(1-p)

    Edge cases:
      - all same class (p=0 or p=1): entropy is 0 (perfectly pure)
      - empty: entropy is 0
    """
    if len(y) == 0:
        return 0.0
    p = float(np.mean(y))
    # TODO: handle the edge cases (p == 0 or p == 1) and compute the formula
    return 0.0


# Sanity checks
print(f"entropy([1,1,1,1]) = {entropy(np.array([1,1,1,1])):.4f}   expect 0.0  (pure)")
print(f"entropy([1,0])     = {entropy(np.array([1,0])):.4f}      expect 1.0  (50/50)")
print(f"entropy(y full)    = {entropy(y):.4f}      expect 0.971")
assert abs(entropy(np.array([1,1,1,1])) - 0.0) < 1e-9
assert abs(entropy(np.array([1,0])) - 1.0) < 1e-9
assert abs(entropy(y) - 0.971) < 0.01
print("✓ entropy() works")


## Exercise 2 — information gain

$$\text{IG}(\text{feature}) = H(\text{parent}) - \bigl[ w_\text{left} H(\text{left}) + w_\text{right} H(\text{right}) \bigr]$$

The weights are the fractions of examples going each way. Higher info gain = a better split.


In [ ]:
def info_gain(X, y, feature):
    """How much entropy decreases if we split on this feature.

    Math:
      H(parent) - [w_left * H(left) + w_right * H(right)]

    where w_left = fraction of examples that go left (feature == 0).
    Higher is better — the feature that gives the biggest info gain wins.
    """
    parent = entropy(y)
    left_mask = X[:, feature] == 0
    n_left = int(left_mask.sum())
    n_right = len(y) - n_left
    if n_left == 0 or n_right == 0:
        return 0.0
    # TODO: compute weighted child entropy and subtract from parent
    return 0.0


# Compute info gain for each of the 3 features
gains = [info_gain(X, y, j) for j in range(3)]
print(f"feature 0 (ear shape):  info gain = {gains[0]:.4f}")
print(f"feature 1 (face round): info gain = {gains[1]:.4f}")
print(f"feature 2 (whiskers):   info gain = {gains[2]:.4f}")
assert gains[0] > gains[1]
assert gains[0] > gains[2]
print(f"\nBest feature to split on first: {int(np.argmax(gains))} (highest info gain)")


## Exercise 3 — recursive tree builder

For each node:
1. If the data is pure or we hit max depth → leaf (predict the majority class).
2. Else, find the feature with highest info gain, split on it, recurse on both sides.

The recursion bottoms out on every path, eventually producing leaves.


In [ ]:
class Node:
    """A tree node — either a split (with feature, left, right) or a leaf (with prediction)."""
    def __init__(self, feature=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.left = left
        self.right = right
        self.prediction = prediction


def build_tree(X, y, max_depth=4, depth=0):
    """Recursively build a decision tree.

    Stop and return a leaf if:
      - y is empty
      - entropy is 0 (pure node — all examples same class)
      - we hit max_depth
      - no feature gives positive info gain
    """
    if len(y) == 0:
        return Node(prediction=0)
    if entropy(y) == 0 or depth >= max_depth:
        return Node(prediction=int(round(np.mean(y))))
    # TODO: pick best feature, return Node(feature=best, left=..., right=...)
    # Hint: compute gains for each feature; if best gain is 0, return a leaf with majority class.
    return Node(prediction=int(round(np.mean(y))))


def predict_one(node, x):
    while node.prediction is None:
        node = node.left if x[node.feature] == 0 else node.right
    return node.prediction


def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])


tree = build_tree(X, y)
y_pred = predict(tree, X)
acc = float((y_pred == y).mean())
print(f"Training accuracy: {acc:.4f}   expected 1.0000 (this dataset is small enough to memorize)")
assert acc == 1.0
print("✓ build_tree() works")


## Visualize the tree

A simple recursive printer with indentation. Each line is either "feature N? (0=left, 1=right)" or "→ predict 0/1".

You should see something like:
```
feature 0?
  left:        # not pointy ears
    feature 2?
      left: → predict 0   (no whiskers, no pointy ears = not cat)
      right: → predict 1
  right:       # pointy ears
    → predict 1   (always cat)
```


In [ ]:
def show(node, depth=0):
    indent = "  " * depth
    if node.prediction is not None:
        print(f"{indent}→ predict {node.prediction}")
    else:
        print(f"{indent}feature {node.feature}? (0=left, 1=right)")
        print(f"{indent}left:")
        show(node.left, depth + 1)
        print(f"{indent}right:")
        show(node.right, depth + 1)


print("Trained decision tree:")
show(tree)


## ⭐ Stretch — bagged forest

Bagging = "Bootstrap AGGregatING". Take N bootstrap samples (sample with replacement, same size as original), train one tree on each, vote at prediction time.

Real random forests *also* randomize which features each split considers, but bagging alone illustrates the idea.

On our 10-row dataset, the bagged forest may actually do **worse** than the single tree (~90% vs 100%) because the data is so small that bootstrap samples miss key examples. Forests' value shows up on large noisy datasets where any single tree would overfit.


In [ ]:
# STRETCH: bagged "forest" — train multiple trees on bootstrap samples, vote.

np.random.seed(0)
n_trees = 5
trees = []
for _ in range(n_trees):
    # TODO: draw a bootstrap sample (np.random.choice with replace=True), build tree, append
    idx = np.arange(len(X))   # placeholder — replace with bootstrap sample
    trees.append(build_tree(X[idx], y[idx]))

# Majority vote
votes = np.array([predict(t, X) for t in trees])
forest_pred = (votes.mean(axis=0) >= 0.5).astype(int)
acc = float((forest_pred == y).mean())
print(f"Bagged forest accuracy: {acc:.4f}")


## Wrap-up

Decision trees are simpler than neural networks, but they're a top-tier tool for tabular data. XGBoost (the same idea + boosting) wins more Kaggle competitions on tabular datasets than any neural-network approach.

What you didn't do: handle continuous features (you'd sort and try every midpoint as a threshold), regression trees (replace entropy with variance), or boosting (sequential trees that focus on prior errors). All extensions of the same `build_tree` skeleton.
